# Oppgave 5

### 2)

Ved å benytte Song-metoden på den Fouriertransformerte Cahn-Hilliard ligningen får vi

$$
\hat{U}^{(1)} = \hat{U}^n + \tau \left[ (\alpha \left| k \right|^2 - \kappa \left| k \right|^4 ) \hat{U}^{(1)} + \widehat{\mathcal{N}(U^n)} \right]
$$

$$
\hat{U}^{(2)} = \alpha_{10} \hat{U}^n + \alpha_{11} \hat{U}^{(1)} + \beta_1 \tau \left[ (\alpha \left| k \right|^2 - \kappa \left| k \right|^4 ) \hat{U}^{(2)} + \widehat{\mathcal{N}(U^{(1)})} \right]
$$

$$
\hat{U}^{(n+1)} = \alpha_{20} \hat{U}^n + \alpha_{21} \hat{U}^{(1)} + \alpha_{22} \hat{U}^{(2)} + \beta_2 \tau \left[ (\alpha \left| k \right|^2 - \kappa \left| k \right|^4 ) \hat{U}^{(n+1)} + \widehat{\mathcal{N}(U^{(2)})} \right]
$$

Hvor $\mathcal{N}(U) = \Delta U^3 - (1 + a) \Delta U$. <br>
Deretter legger vi til $\tau \beta_i \hat{g}(x, y, t + \frac{\tau}{2})$ på høyresiden. Ved å løse for de ukjente får vi

$$
\hat{U}^{(1)} = \frac{\hat{U}^n + \tau \widehat{\mathcal{N}(U^n)} + \tau \hat{g}(x, y, t + \frac{\tau}{2})}{(1 + \tau (\kappa \left| k \right|^4 - \alpha \left| k \right|^2)) }
$$
  
$$
\hat{U}^{(2)} = \frac{\alpha_{10} \hat{U}^n + \alpha_{11} \hat{U}^{(1)} + \beta_1 \tau \widehat{\mathcal{N}(U^{(1)})} + \beta_1 \tau \hat{g}(x, y, t + \frac{\tau}{2})}{(1 + \beta_1 \tau (\kappa \left| k \right|^4 - \alpha \left| k \right|^2)) }
$$
  
$$
\hat{U}^{(n+1)} = \frac{\alpha_{20} \hat{U}^n + \alpha_{21} \hat{U}^{(1)} + \alpha_{22} \hat{U}^{(2)} + \beta_2 \tau \widehat{\mathcal{N}(U^{(2)})} + \beta_2 \tau \hat{g}(x, y, t + \frac{\tau}{2})}{(1 + \beta_2 \tau (\kappa \left| k \right|^4 - \alpha \left| k \right|^2))}
$$



In [2]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fft2, ifft2, fftfreq, fftshift

def IMEX_solver(*, 
                                kappa, 
                                X, Y, U0, 
                                t0, T, Nt,
                                g, coeff,
                                alpha=1.5):
    """
    Implements the Cahn-Hilliard equation solver using the backward Euler method 
    with a convex-concave splitting approach.

    Parameters:
    -----------
    kappa : float
        Diffusion coefficient for the biharmonic operator.
    X : ndarray
        2D array representing the x-coordinates of the grid.
    Y : ndarray
        2D array representing the y-coordinates of the grid.
    U0 : ndarray
        Initial condition for the solution.
    t0 : float
        Initial time.
    T : float
        Final time.
    Nt : int
        Number of time steps.
    g : callable or None
        Source term as a function of (X, Y, t). If None, no source term is applied.
    alpha : float, optional
        Convex-concave splitting parameter. Default is 1.5.

    Yields:
    -------
    tuple: A tuple containing the discrete Fourier transform of U at t, and the current time t.

    """

    # Definerer relevant data for Fouriertransformen
    N, M = U0.shape
    dx = X[0, 1] - X[0, 0]
    dy = Y[1, 0] - Y[0, 0]
    k_x = (fftfreq(N, d=dx/(2*np.pi)))
    k_y = (fftfreq(M, d=dy/(2*np.pi)))
    alpha_10, alpha_11, alpha_20, alpha_21, alpha_22, beta_1, beta_2 = coeff  # Henter ut koeffisientene

    KX, KY = np.meshgrid(k_x, k_y, indexing="ij")
    K2 = -(KX**2 + KY**2)
    K4 = K2**2

    # Beregner diskret Fouriertransform av initialverdi
    U_hat = fft2(U0)

    # Beregner tidssteg
    t = t0 
    dt = (T-t0)/Nt

    yield U_hat, t  

    
    while t < T-dt/2:
        
        # Behandler funksjonsparameter g 
        if g is not None:
            g_hat = fft2(g(X, Y, t + dt/2, kappa))
        else:
            g_hat = 0
        
        def NonLinear(U_hat):
            U = ifft2(U_hat).real
            NonLin = U**3 - (1 + alpha) * U
            return K2*fft2(NonLin)

        # Bruker Song-metoden til å finne neste steg
        U_hat_1 = (U_hat + dt * (NonLinear(U_hat) + g_hat)) / (1 + dt * (kappa * K4 - alpha * K2))
        U_hat_2 = (alpha_10 * U_hat + alpha_11 * U_hat_1 + beta_1 * dt * (NonLinear(U_hat_1) + g_hat)) / (1 + beta_1 * dt * (kappa * K4 - alpha * K2))
        U_hat_next = (alpha_20 * U_hat + alpha_21 * U_hat_1 + alpha_22 * U_hat_2 + beta_2 * dt * (NonLinear(U_hat_2) + g_hat)) / (1 + beta_2 * dt * (kappa * K4 - alpha * K2))
       
        # Oppdaterer U_hat og t
        U_hat = U_hat_next
        t += dt

        yield U_hat, t

In [3]:
# Definerer eksakt løsning
def U_ex(x, y, t, kappa):
    return np.sin(x) * np.cos(y) * np.exp(-4*kappa*t)

# Bruker sympy til å beregne høyresiden g til likningen
x, y, t, kappa = sp.symbols("x y t kappa")
u_ex = sp.sin(x) * sp.cos(y) * sp.exp(-4*kappa*t)
laplacian_u = sp.diff(u_ex, x, 2) + sp.diff(u_ex, y, 2)
biharmonic_u = sp.diff(laplacian_u, x, 2) + sp.diff(laplacian_u, y, 2)
nonlinear = sp.diff(u_ex**3 - u_ex, x, 2) + sp.diff(u_ex**3 - u_ex, y, 2)
g = sp.diff(u_ex, t) + kappa * biharmonic_u - nonlinear
g_simplified = sp.simplify(g)

# Beregner høyresiden g for 
g_lambdify001 = sp.lambdify((x, y, t, kappa), g_simplified, "numpy")
g_func = lambda X, Y, t, kappa: g_lambdify001(X, Y, t, kappa)

In [4]:
t0, T = 0, 1
Nts = [100, 200, 400, 800, 1600, 3200]
kappa = 0.01

N = 64
x, y = np.linspace(0, 16*np.pi, N, endpoint=False), np.linspace(0, 16*np.pi, N, endpoint=False)
X, Y = np.meshgrid(x, y)

# Definerer 4 sett med koeffisienter
coeff_1 = (3/2, -1/2, 0, 0, 1, 1/2, 1)
coeff_2 = (2, -1, 1/2, 0, 1/2, 1, 1)
coeff_3 = (2, -1, 0, 1/2, 1/2, 1, 1/2)
coeff_4 = (5/2, -3/2, 2/3, 0, 1/3, 3/2, 1)

for coeff_indeks, coeff in zip([1, 2, 3, 4], [coeff_1, coeff_2, coeff_3, coeff_4]):
    
    U0 = U_ex(X, Y, 0, kappa)
    errors = []                     # Liste til feilverdier
    hs = []                         # Liste til korresponderende steglengde

    for Nt in Nts:
        dt = (T - t0) / Nt
        solver = IMEX_solver(kappa=kappa, X=X, Y=Y, U0=U0, t0=t0, T=T, Nt=Nt, g = g_func, alpha=1.5, coeff = coeff)
        max_error = 0

        for Uhat, t in solver:
            U = ifft2(Uhat).real
            new_error = np.max(np.abs(U - U_ex(X, Y, t, kappa)))    # Beregner feilen mellom numerisk løsning og eksakt løsning

            if new_error > max_error:
                max_error = new_error
        
        errors.append(max_error)
        hs.append(dt)

    # Beregner EOC
    eocs = []                       # Liste til EOC-verdier

    for i in range(1, len(errors)):
        eoc = np.log(errors[i] / errors[i-1]) / np.log(hs[i] / hs[i-1])     # Beregner EOC
        eocs.append(eoc)                                                    # Legger til EOC i liste

    # Printer resultater i tabell
    print(f"\na_10 = {coeff[0]}, a_11 = {coeff[1]}, a_20 = {coeff[2]}, a_21 = {coeff[3]}, a_22 = {coeff[4]}, b_1 = {coeff[5]}, b_2 = {coeff[6]}")
    print(f"Konvergenstabell for κ = {kappa}:")
    print(f"{'Nt':<10}{'Max Error':<20}{'EOC':<10}")
    print("-" * 40)
    for i, Nt in enumerate(Nts):
        if i == 0:
            print(f"{Nt:<10}{errors[i]:<20.2e}{'-':<10}")
        else:
            print(f"{Nt:<10}{errors[i]:<20.2e}{eocs[i-1]:<10.2f}")


a_10 = 1.5, a_11 = -0.5, a_20 = 0, a_21 = 0, a_22 = 1, b_1 = 0.5, b_2 = 1
Konvergenstabell for κ = 0.01:
Nt        Max Error           EOC       
----------------------------------------
100       8.37e-05            -         
200       2.16e-05            1.96      
400       5.41e-06            2.00      
800       1.35e-06            2.01      
1600      3.35e-07            2.01      
3200      8.37e-08            2.00      

a_10 = 2, a_11 = -1, a_20 = 0.5, a_21 = 0, a_22 = 0.5, b_1 = 1, b_2 = 1
Konvergenstabell for κ = 0.01:
Nt        Max Error           EOC       
----------------------------------------
100       2.00e-05            -         
200       8.93e-06            1.16      
400       2.95e-06            1.60      
800       8.43e-07            1.81      
1600      2.25e-07            1.91      
3200      5.81e-08            1.95      

a_10 = 2, a_11 = -1, a_20 = 0, a_21 = 0.5, a_22 = 0.5, b_1 = 1, b_2 = 0.5
Konvergenstabell for κ = 0.01:
Nt        Max Error         

Tabellene over viser en oversikt over maksimal feil og EOC avhengig av antall tidssteg $N_t$ for fire ulike sett med Song-koeffisienter. 

Vi ser at Song-metoden har en eksperimentell konvergensorden på $p = 2$ for alle koeffisientene. I oppgave 4 så vi at implisitt Euler metode hadde en EOC på $p = 1$. Dette impliserer at Song-metoden er betraktelig mer effektiv enn implisitt Euler, da den samme feiltoleransen vil kunne oppnås med færre tidssteg ved å ta i bruk Song-metoden. 

Vi ser at det første settet med Song-koeffisienter, 
\begin{aligned}
\alpha_{10} = \frac{3}{2}, \quad \alpha_{11} = - \frac{1}{2}, \quad \alpha_{20} = 0, \quad \alpha_{21} = 0, \quad \alpha_{22} = 1, \quad \beta_1 = - \frac{1}{2}, \quad \beta_2 = 1
\end{aligned}

gir høyest og raskest konvergerende eksperimentell konvergensorden. Disse koeffisientene vil altså oppfylle Song-metodens potensiale raskere enn de andre. 

I neste oppgave vil vi derfor benytte Song-metoden med det første settet med Song-koeffisienter for å løse Cahn-Hilliard likningen.